# CIV Best-Model Monte Carlo Uncertainty Workflow

This notebook performs exactly one `MC=True` fit (`nsamp=100`) per target, using the best A–D model already selected in `Target_lists/civ_output_final_target_values.csv`. The adopted central C IV measurements and existing MCMC uncertainties are copied unchanged from that table; the noisy-refit dispersions are saved as new `*_err_mc` columns in a separate CSV. The existing MCMC products are never overwritten.


In [1]:
%matplotlib inline
from pathlib import Path
import ast
import re
import tempfile
import timeit
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table

import sys
sys.path.append('../')
from pyqsofit.PyQSOFit import QSOFit

warnings.filterwarnings('ignore')
QSOFit.set_mpl_style()

In [2]:
# Input/output configuration
PROJECT_ROOT = Path('/Users/nayera/PyQSOFit')
BEST_MODEL_CSV = PROJECT_ROOT / 'Target_lists' / 'civ_output_final_target_values.csv'
TRACKING_CSV = PROJECT_ROOT / 'Target_lists' / 'civ_batch_mc_fit_inputs_1600.csv'
RESULTS_CSV = PROJECT_ROOT / 'Target_lists' / 'civ_output_final_target_values_mc.csv'

# Separate Monte Carlo run: PyQSOFit does not allow MC and MCMC together.
RUN_MC = True
RUN_MCMC = False
NSAMP = 100

# Match the original fit configuration.
SNR_THRESHOLD = 3.0
LOW_SNR_ERROR_SCALE = 1e6
MAX_CIV_FWHM_KMS_SIGMA = 0.02157
NARROW_FWHM_LIMIT_KMS = 1600
NARROW_CIV_MAXSIG = 0.0022664260627611094
BROAD_FWHM_THRESHOLD = NARROW_FWHM_LIMIT_KMS
LAM_LAB_CIV = 1549.06
C_KMS = 299792.458

MODEL_GRID = {
    'A': {'br_component': 3, 'na_component': 0},
    'B': {'br_component': 2, 'na_component': 1},
    'C': {'br_component': 2, 'na_component': 0},
    'D': {'br_component': 1, 'na_component': 1},
}

# Set to a small integer only for a future smoke test; leave None for all selected targets.
TARGET_LIMIT = None


In [3]:
def as_serializable(value):
    if value is None:
        return ''
    return np.asarray(value).tolist()


def parse_saved_array(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, (list, tuple, np.ndarray)):
        arr = np.asarray(value, dtype=float)
    else:
        text = str(value).strip()
        if not text or text.lower() in {'nan', 'none', ''}:
            return None
        arr = np.asarray(ast.literal_eval(text), dtype=float)
    return arr


def load_best_models(best_model_csv=BEST_MODEL_CSV):
    df = pd.read_csv(best_model_csv)
    required = {
        'Target', 'Model', 'file_name', 'fits_path', 'fit_z', 'fit_z_source',
        'range_arr', 'mask_arr', 'br_components_forced', 'na_components_forced',
        'lam_blue', 'lam_red', 'lam_half', 'br_peak', 'FWHM_CIV',
        'FWHM_err_mcmc', 'CIV_blueshift_kms', 'CIV_asym',
    }
    missing = sorted(required.difference(df.columns))
    if missing:
        raise ValueError(f'Missing required columns in {best_model_csv.name}: {missing}')
    if df['Target'].duplicated().any():
        duplicated = df.loc[df['Target'].duplicated(False), 'Target'].tolist()
        raise ValueError(f'Expected one selected model per target; duplicates: {duplicated}')

    df['Model'] = df['Model'].astype(str).str.strip().str.upper()
    invalid_models = sorted(set(df['Model']).difference(MODEL_GRID))
    if invalid_models:
        raise ValueError(f'Unknown selected models: {invalid_models}')

    for idx, row in df.iterrows():
        expected = MODEL_GRID[row['Model']]
        actual = (int(row['br_components_forced']), int(row['na_components_forced']))
        wanted = (expected['br_component'], expected['na_component'])
        if actual != wanted:
            raise ValueError(
                f"{row['Target']} model {row['Model']} has component counts {actual}, expected {wanted}"
            )

    df['target_name'] = df['Target'].astype(str).str.strip()
    df['range_arr_parsed'] = df['range_arr'].apply(parse_saved_array)
    df['mask_arr_parsed'] = df['mask_arr'].apply(parse_saved_array)
    df['fits_exists'] = df['fits_path'].apply(lambda value: Path(value).exists())
    df['valid_fit_z'] = pd.to_numeric(df['fit_z'], errors='coerce').notna()
    df['valid_range'] = df['range_arr_parsed'].apply(lambda value: value is not None and value.size == 2)
    df['run_fit'] = df['fits_exists'] & df['valid_fit_z'] & df['valid_range']

    reasons = []
    for row in df.itertuples(index=False):
        missing = []
        if not row.fits_exists:
            missing.append('missing FITS file')
        if not row.valid_fit_z:
            missing.append('missing/non-numeric fit redshift')
        if not row.valid_range:
            missing.append('missing/invalid continuum fit range')
        reasons.append('; '.join(missing))
    df['skip_reason_mc'] = reasons
    df.to_csv(TRACKING_CSV, index=False)
    return df


selected_targets = load_best_models()
fit_targets = selected_targets[selected_targets['run_fit']].copy()
if TARGET_LIMIT is not None:
    fit_targets = fit_targets.head(TARGET_LIMIT).copy()

print(f'Loaded {len(selected_targets)} selected best-model rows from {BEST_MODEL_CSV.name}.')
print(f'{len(fit_targets)} targets are ready for one MC fit each (nsamp={NSAMP}).')
print(f'Tracking table will be written to {TRACKING_CSV}.')
if (~selected_targets['run_fit']).any():
    display(selected_targets.loc[~selected_targets['run_fit'], ['Target', 'Model', 'skip_reason_mc']])
display(fit_targets[['Target', 'Model', 'br_components_forced', 'na_components_forced', 'fits_path']])


Loaded 14 selected best-model rows from civ_output_final_target_values.csv.
14 targets are ready for one MC fit each (nsamp=100).
Tracking table will be written to /Users/nayera/PyQSOFit/Target_lists/civ_batch_mc_fit_inputs_1600.csv.


,Target,Model,br_components_forced,na_components_forced,fits_path
0,J1245+4348,C,2,0,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
1,J1100+5800,D,1,1,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
2,J1328+4445,C,2,0,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
3,J0831+4046,C,2,0,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
4,J1102+6635,C,2,0,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
5,J1342+5838,C,2,0,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
6,J1534+5711,D,1,1,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
7,J0941+5947,C,2,0,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
8,J0759+1800,C,2,0,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
9,J0807+1328,A,3,0,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...


In [4]:
def continuum_windows_from_range(range_arr):
    lo, hi = map(float, range_arr)
    low_start = max(lo, 1445.0)
    low_end = min(low_start + 20.0, hi)
    high_end = hi
    high_start = max(hi - 15.0, low_end + 5.0)
    if low_end - low_start < 5.0 or high_end - high_start < 5.0 or high_start <= low_end:
        width = hi - lo
        low_start, low_end = lo, lo + 0.15 * width
        high_start, high_end = hi - 0.15 * width, hi
    return np.rec.array(
        [(low_start, low_end), (high_start, high_end)],
        formats='float32,float32',
        names='min,max',
    )


def make_qsopar(param_dir, range_arr, br_component, na_component):
    primary_hdu = fits.PrimaryHDU(header=fits.Header({'Author': 'Nayera Abdessalam'}))

    line_priors = np.rec.array([
        (1549.06, 'CIV', 1450, 1700, 'CIV_br', br_component, 0.0, 0.0, 1e10,
         0.005, 0.004, MAX_CIV_FWHM_KMS_SIGMA, 0.015, 0, 0, 0, 0.05, 1),
        (1549.06, 'CIV', 1450, 1700, 'CIV_na', na_component, 0.0, 0.0, 1e10,
         1e-3, 5e-4, NARROW_CIV_MAXSIG, 0.01, 1, 1, 0, 0.002, 1),
    ], formats='float32,a20,float32,float32,a20,int32,float32,float32,float32,float32,float32,float32,float32,int32,int32,int32,float32,int32',
       names='lambda,compname,minwav,maxwav,linename,ngauss,inisca,minsca,maxsca,inisig,minsig,maxsig,voff,vindex,windex,findex,fvalue,vary')
    hdu1 = fits.BinTableHDU(data=line_priors, name='line_priors')

    hdu2 = fits.BinTableHDU(data=continuum_windows_from_range(range_arr), name='conti_windows')

    conti_priors = np.rec.array([
        ('Fe_uv_norm', 0.0, 0.0, 1e10, 1),
        ('Fe_uv_FWHM', 3000, 1200, 18000, 1),
        ('Fe_uv_shift', 0.0, -0.01, 0.01, 1),
        ('Fe_op_norm', 0.0, 0.0, 1e10, 1),
        ('Fe_op_FWHM', 3000, 1200, 18000, 1),
        ('Fe_op_shift', 0.0, -0.01, 0.01, 1),
        ('PL_norm', 1.0, 0.0, 1e10, 1),
        ('PL_slope', -1.5, -5.0, 3.0, 1),
        ('Blamer_norm', 0.0, 0.0, 1e10, 1),
        ('Balmer_Te', 15000, 10000, 50000, 1),
        ('Balmer_Tau', 0.5, 0.1, 2.0, 1),
        ('conti_a_0', 0.0, None, None, 1),
        ('conti_a_1', 0.0, None, None, 1),
        ('conti_a_2', 0.0, None, None, 1),
    ], formats='a20,float32,float32,float32,int32', names='parname,initial,min,max,vary')
    hdu3 = fits.BinTableHDU(data=conti_priors, name='conti_priors')

    measure_info = Table([[[1350, 1450, 3000, 4200, 5100]], [[[1500, 1550]]]],
                         names=['cont_loc', 'Fe_flux_range'], dtype=['float32', 'float32'])
    hdu4 = fits.BinTableHDU(data=measure_info, name='measure_info')

    param_path = Path(param_dir) / 'qsopar.fits'
    fits.HDUList([primary_hdu, hdu1, hdu2, hdu3, hdu4]).writeto(param_path, overwrite=True)
    return param_path


In [5]:
def read_sdss_spectrum(fits_path):
    with fits.open(fits_path) as data:
        lam = 10 ** data[1].data['loglam']
        flux = np.asarray(data[1].data['flux'], dtype=float)
        ivar = np.asarray(data[1].data['ivar'], dtype=float)
        err = np.full_like(flux, np.inf, dtype=float)
        good_ivar = np.isfinite(ivar) & (ivar > 0)
        err[good_ivar] = 1 / np.sqrt(ivar[good_ivar])
        header = data[0].header
        meta = {
            'ra': header.get('plug_ra', np.nan), 'dec': header.get('plug_dec', np.nan),
            'plateid': header.get('plateid', -1), 'mjd': header.get('mjd', -1),
            'fiberid': header.get('fiberid', -1),
        }
    good = np.isfinite(flux) & np.isfinite(err) & (err > 0)
    snr = np.zeros_like(flux, dtype=float)
    snr[good] = np.abs(flux[good]) / err[good]
    low_snr = snr < SNR_THRESHOLD
    err[low_snr] *= LOW_SNR_ERROR_SCALE
    meta['snr_masked_pixels'] = int(np.sum(low_snr))
    meta['total_pixels'] = int(len(flux))
    meta['snr_masked_fraction'] = float(np.sum(low_snr) / len(flux))
    return lam, flux, err, meta


def get_line_value(q, key, default=np.nan):
    idx = np.where(np.asarray(q.line_result_name) == key)[0]
    if idx.size == 0:
        return default
    return float(np.ravel(q.line_result[idx[0]])[0])


def sample_gaussian_params(q, line_name, sample_index=None):
    result_names = np.asarray(q.gauss_result_name)
    params = []
    component = 1
    while True:
        keys = [
            f'{line_name}_{component}_scale',
            f'{line_name}_{component}_centerwave',
            f'{line_name}_{component}_sigma',
        ]
        masks = [result_names == key for key in keys]
        if not all(np.any(mask) for mask in masks):
            break
        if sample_index is None:
            result = np.asarray(q.gauss_result, dtype=object)
            values = [float(np.ravel(result[mask][0])[0]) for mask in masks]
        else:
            # gauss_result_name interleaves value/error; gauss_result_all contains values only.
            indices = [int(np.flatnonzero(mask)[0] // 2) for mask in masks]
            values = [float(q.gauss_result_all[sample_index, index]) for index in indices]
        params.append(values)
        component += 1
    return np.asarray(params, dtype=float)


def profile_measurements(q, sample_index=None):
    params = sample_gaussian_params(q, 'CIV_br', sample_index=sample_index)
    if params.size == 0:
        return dict(lam_blue=np.nan, lam_red=np.nan, lam_half=np.nan, br_peak=np.nan)
    lam = np.asarray(q.wave, dtype=float)
    selected = (lam >= 1450.0) & (lam <= 1700.0)
    if np.sum(selected) < 3:
        selected = np.ones_like(lam, dtype=bool)
    lam_w = lam[selected]
    lnlam = np.log(lam_w)
    model = np.zeros_like(lam_w, dtype=float)
    for amplitude, center, sigma in params:
        model += amplitude * np.exp(-0.5 * ((lnlam - center) / sigma) ** 2)
    if not np.any(np.isfinite(model)) or np.nanmax(model) <= 0:
        return dict(lam_blue=np.nan, lam_red=np.nan, lam_half=np.nan, br_peak=np.nan)

    peak_flux = np.nanmax(model)
    peak_index = int(np.nanargmax(model))
    br_peak = float(lam_w[peak_index])
    half_peak = 0.5 * peak_flux

    left_below = np.where(model[:peak_index] < half_peak)[0]
    if left_below.size == 0:
        lam_blue = float(lam_w[0])
    else:
        j0 = left_below[-1]
        lam_blue = float(np.interp(half_peak, model[j0:j0 + 2], lam_w[j0:j0 + 2]))

    right_below = np.where(model[peak_index + 1:] < half_peak)[0]
    if right_below.size == 0:
        lam_red = float(lam_w[-1])
    else:
        j1 = peak_index + 1 + right_below[0]
        j0 = j1 - 1
        lam_red = float(np.interp(half_peak, model[j0:j1 + 1], lam_w[j0:j1 + 1]))

    increments = 0.5 * (model[1:] + model[:-1]) * np.diff(lam_w)
    cumulative = np.cumsum(increments)
    lam_half = np.nan if cumulative.size == 0 or cumulative[-1] <= 0 else float(
        np.interp(0.5 * cumulative[-1], cumulative, lam_w[1:])
    )
    return dict(lam_blue=lam_blue, lam_red=lam_red, lam_half=lam_half, br_peak=br_peak)


def blueshift_calc(lam_half):
    return C_KMS * (LAM_LAB_CIV - lam_half) / LAM_LAB_CIV


def asym_calc(lam_blue, lam_red, lam_peak):
    numerator = lam_red - lam_peak
    denominator = lam_peak - lam_blue
    if numerator <= 0 or denominator <= 0:
        return np.nan
    return float(np.log(numerator / denominator))


def mc_dispersion(values):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    return float(np.std(finite, ddof=1)) if finite.size > 1 else np.nan


def collect_mc_uncertainties(q):
    n_samples = len(q.gauss_result_all)
    profiles = [profile_measurements(q, sample_index=i) for i in range(n_samples)]
    lam_blue = np.array([item['lam_blue'] for item in profiles])
    lam_red = np.array([item['lam_red'] for item in profiles])
    lam_half = np.array([item['lam_half'] for item in profiles])
    # PyQSOFit's line-property calculation supplies the continuous-model peak wavelength.
    # Use it instead of the wavelength-grid peak for the Zuo asymmetry calculation.
    broad_properties = np.array([
        q.line_prop_from_name('CIV_br', 'broad', sample_index=i, ln_sigma_br=NARROW_CIV_MAXSIG)
        for i in range(n_samples)
    ], dtype=float)
    fwhm = broad_properties[:, 0]
    br_peak = broad_properties[:, 3]
    blueshift = np.array([blueshift_calc(value) for value in lam_half])
    asymmetry = np.array([
        asym_calc(blue, red, peak) for blue, red, peak in zip(lam_blue, lam_red, br_peak)
    ])
    return {
        'FWHM_CIV_err_mc': mc_dispersion(fwhm),
        'CIV_blueshift_err_mc': mc_dispersion(blueshift),
        'CIV_asym_err_mc': mc_dispersion(asymmetry),
        'lam_half_err_mc': mc_dispersion(lam_half),
        'lam_blue_err_mc': mc_dispersion(lam_blue),
        'lam_red_err_mc': mc_dispersion(lam_red),
        'br_peak_err_mc': mc_dispersion(br_peak),
        'mc_samples_requested': NSAMP,
        'mc_samples_returned': n_samples,
        'mc_valid_blueshift_samples': int(np.isfinite(blueshift).sum()),
        'mc_valid_asym_samples': int(np.isfinite(asymmetry).sum()),
    }


In [6]:
def fit_target_best_model(target_row):
    target = target_row['Target']
    model_name = target_row['Model']
    model_cfg = MODEL_GRID[model_name]
    fits_path = Path(target_row['fits_path'])
    z = float(target_row['fit_z'])
    range_arr = np.asarray(target_row['range_arr_parsed'], dtype=float)
    mask_arr = target_row['mask_arr_parsed']
    if mask_arr is not None:
        mask_arr = np.asarray(mask_arr, dtype=float)

    lam, flux, err, meta = read_sdss_spectrum(fits_path)
    with tempfile.TemporaryDirectory(prefix='pyqsofit_civ_mc_') as param_dir:
        make_qsopar(param_dir, range_arr, model_cfg['br_component'], model_cfg['na_component'])
        q = QSOFit(lam, flux, err, z,
                   ra=meta['ra'], dec=meta['dec'], plateid=meta['plateid'],
                   mjd=meta['mjd'], fiberid=meta['fiberid'], path=param_dir)
        start = timeit.default_timer()
        q.Fit(name=f'{target}_{model_name}_mc',
              nsmooth=1,
              and_mask=False,
              or_mask=False,
              reject_badpix=True,
              deredden=False,
              wave_range=range_arr,
              wave_mask=mask_arr,
              decompose_host=False,
              host_prior=False,
              host_prior_scale=0.2,
              host_line_mask=True,
              decomp_na_mask=True,
              qso_type='CZBIN1',
              npca_qso=10,
              host_type='PCA',
              npca_gal=5,
              Fe_uv_op=False,
              poly=False,
              BC=False,
              initial_guess=None,
              rej_abs_conti=True,
              n_pix_min_conti=100,
              linefit=True,
              rej_abs_line=True,
              MC=RUN_MC,
              MCMC=RUN_MCMC,
              nsamp=NSAMP,
              param_file_name='qsopar.fits',
              epsilon_jitter=0.,
              save_result=False,
              plot_fig=False,
              save_fig=False,
              plot_corner=False,
              verbose=False,
              kwargs_plot={'broad_fwhm': BROAD_FWHM_THRESHOLD},
              kwargs_conti_emcee={},
              kwargs_line_emcee={})
        fit_seconds = float(timeit.default_timer() - start)

    # Start with the selected-table row so adopted central values and MCMC errors remain exact.
    row = target_row.drop(labels=[
        'target_name', 'range_arr_parsed', 'mask_arr_parsed', 'fits_exists',
        'valid_fit_z', 'valid_range', 'run_fit', 'skip_reason_mc',
    ], errors='ignore').to_dict()
    row.update(collect_mc_uncertainties(q))
    row.update({
        'mc_model': model_name,
        'mc_br_components_forced': model_cfg['br_component'],
        'mc_na_components_forced': model_cfg['na_component'],
        'mc_fit_red_chi2': get_line_value(q, '1_line_red_chi2'),
        'mc_fit_bic': get_line_value(q, '1_line_bic'),
        'mc_fit_seconds': fit_seconds,
        'mc_status': 'ok',
        'mc_error': '',
    })
    return row


In [7]:
results = []
expected = len(fit_targets)
print(f'Prepared to run {expected} fits: one selected best-model MC fit per target, nsamp={NSAMP}.')

for _, target_row in fit_targets.iterrows():
    label = f"{target_row['Target']} model {target_row['Model']}"
    print(f'Starting {label}')
    try:
        row = fit_target_best_model(target_row)
        print(
            f"Finished {label}: FWHM_err_mc={row['FWHM_CIV_err_mc']:.3g} km/s, "
            f"blueshift_err_mc={row['CIV_blueshift_err_mc']:.3g} km/s"
        )
    except Exception as exc:
        row = target_row.drop(labels=[
            'target_name', 'range_arr_parsed', 'mask_arr_parsed', 'fits_exists',
            'valid_fit_z', 'valid_range', 'run_fit', 'skip_reason_mc',
        ], errors='ignore').to_dict()
        row.update({
            'mc_model': target_row['Model'],
            'mc_br_components_forced': int(target_row['br_components_forced']),
            'mc_na_components_forced': int(target_row['na_components_forced']),
            'mc_samples_requested': NSAMP,
            'mc_status': 'failed',
            'mc_error': repr(exc),
        })
        print(f'FAILED {label}: {exc!r}')
    results.append(row)
    # Separate checkpoint file; the source best-model/MCMC table is never modified.
    pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)

results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_CSV, index=False)
print(f'Wrote {len(results_df)} rows to {RESULTS_CSV}')
display(results_df)


Prepared to run 14 fits: one selected best-model MC fit per target, nsamp=100.
Starting J1245+4348 model C
Finished J1245+4348 model C: FWHM_err_mc=2.51e+03 km/s, blueshift_err_mc=623 km/s
Starting J1100+5800 model D
Finished J1100+5800 model D: FWHM_err_mc=247 km/s, blueshift_err_mc=111 km/s
Starting J1328+4445 model C
Finished J1328+4445 model C: FWHM_err_mc=37.1 km/s, blueshift_err_mc=89.4 km/s
Starting J0831+4046 model C
Finished J0831+4046 model C: FWHM_err_mc=1.17e+03 km/s, blueshift_err_mc=219 km/s
Starting J1102+6635 model C
Finished J1102+6635 model C: FWHM_err_mc=3.21e+03 km/s, blueshift_err_mc=478 km/s
Starting J1342+5838 model C
Finished J1342+5838 model C: FWHM_err_mc=1.57e+03 km/s, blueshift_err_mc=375 km/s
Starting J1534+5711 model D
Finished J1534+5711 model D: FWHM_err_mc=1.22e+03 km/s, blueshift_err_mc=455 km/s
Starting J0941+5947 model C
Finished J0941+5947 model C: FWHM_err_mc=226 km/s, blueshift_err_mc=107 km/s
Starting J0759+1800 model C
Finished J0759+1800 model 

,Target,Model,file_name,fits_path,z_o3,Wu_z,fit_z,fit_z_source,range_arr,mask_arr,...,mc_valid_blueshift_samples,mc_valid_asym_samples,mc_model,mc_br_components_forced,mc_na_components_forced,mc_fit_red_chi2,mc_fit_bic,mc_fit_seconds,mc_status,mc_error
0,J1245+4348,C,01spec-6617-56365-0860.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.890,4.820,4.890,z (o3),"[1445.0, 1600.0]","[[1495.0, 1503.0]]",...,100,100,C,2,0,0.517198,-233.184035,6.990475,ok,
1,J1100+5800,D,02spec-8184-57426-0330.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.759,4.776,4.759,z (o3),"[1445.0, 1615.0]","[[1502.0, 1513.0], [1533.0, 1537.0]]",...,100,100,D,1,1,0.482990,-274.034293,4.393263,ok,
2,J1328+4445,C,03spec-6627-56369-0578.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.820,4.810,4.820,z (o3),"[1445.0, 1705.0]","[[1499.0, 1510.0], [1526.0, 1528.0], [1565.0, ...",...,100,100,C,2,0,0.649724,-239.436909,11.856241,ok,
3,J0831+4046,C,04spec-3808-55513-0180.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.900,4.820,4.900,z (o3),"[1445.0, 1660.0]","[[1511.0, 1514.0], [1528.0, 1531.0]]",...,100,100,C,2,0,0.666237,-186.398644,7.044595,ok,
4,J1102+6635,C,05spec-7111-56741-0438.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.787,4.803,4.787,z (o3),"[1400.0, 1600.0]","[[1534.0, 1544.0], [1605.0, 1650.0]]",...,100,100,C,2,0,0.313198,-431.027537,6.951072,ok,
5,J1342+5838,C,11spec-6810-56447-0055.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.850,4.855,4.850,z (o3),"[1400.0, 1600.0]",NaN,...,100,100,C,2,0,0.347064,-418.440774,8.612768,ok,
6,J1534+5711,D,12spec-0615-52347-0094.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,NaN,4.820,4.820,Wu_z,"[1400.0, 1580.0]","[[1500.0, 1503.0], [1535.0, 1538.0]]",...,100,100,D,1,1,0.100179,-783.035236,77.241548,ok,
7,J0941+5947,C,17spec-8194-57401-0548.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.840,4.861,4.840,z (o3),"[1445.0, 1675.0]","[[1504.0, 1512.0], [1544.0, 1549.0]]",...,100,100,C,2,0,0.646922,-214.510489,6.014660,ok,
8,J0759+1800,C,20spec-4490-55629-0074.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.796,4.778,4.796,z (o3),"[1425.0, 1600.0]","[[1510.0, 1525.0], [1528.0, 1533.0]]",...,100,100,C,2,0,0.671821,-114.309757,5.549189,ok,
9,J0807+1328,A,21spec-4505-55603-0810.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.880,4.870,4.880,z (o3),"[1450.0, 1650.0]",NaN,...,100,100,A,3,0,0.800818,-74.657894,25.308016,ok,


# STOP

The MC execution cell above is intentionally unrun. Review the configuration, then run the notebook when ready. Output goes only to `Target_lists/civ_output_final_target_values_mc.csv`; the existing best-model/MCMC CSV is not overwritten.
